# Gold Layer - Retail Data Engineering Project

## Purpose
This notebook create business-ready aggregated tables such as sales by product, store, date, and customer.

In [0]:
# Gold Layer Configuration

storage_account = "saretailsneh"

base_path = f"abfss://retail@{storage_account}.dfs.core.windows.net"

silver_base_path = f"{base_path}/silver"
gold_base_path = f"{base_path}/gold"

print("Silver path:", silver_base_path)
print("Gold path:", gold_base_path)

Silver path: abfss://retail@saretailsneh.dfs.core.windows.net/silver
Gold path: abfss://retail@saretailsneh.dfs.core.windows.net/gold


###Load all Silver tables

In [0]:
# Load Silver Tables

customers_silver = spark.read.parquet(
    f"{silver_base_path}/customers/"
)

products_silver = spark.read.parquet(
    f"{silver_base_path}/products/"
)

stores_silver = spark.read.parquet(
    f"{silver_base_path}/stores/"
)

transactions_silver = spark.read.parquet(
    f"{silver_base_path}/transactions/"
)

print("✓ All Silver tables loaded successfully")

✓ All Silver tables loaded successfully


###Verify Silver row counts

In [0]:
# Verify Silver Table Row Counts

print("SILVER TABLE ROW COUNTS")
print("=" * 40)

print(f"Customers:    {customers_silver.count():,}")
print(f"Products:     {products_silver.count():,}")
print(f"Stores:       {stores_silver.count():,}")
print(f"Transactions: {transactions_silver.count():,}")

SILVER TABLE ROW COUNTS
Customers:    5,000
Products:     500
Stores:       50
Transactions: 100,005


###Product Sales Summary

Business table showing sales by product.

In [0]:
from pyspark.sql.functions import sum, round

product_sales = (
    transactions_silver
    .groupBy("product_id")
    .agg(
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 2).alias("total_sales"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_amount"), 2).alias("net_sales")
    )
)

product_sales = (
    product_sales.join(
        products_silver,
        on="product_id",
        how="left"
    )
)

product_sales.display()

product_id,total_quantity,total_sales,total_discount,net_sales,product_name,category,brand,unit_price,last_modified_date,silver_processed_at
343,556,3573033.92,230383.63,3342650.29,Orbit Product 343,Beauty,Aura,6426.32,2026-01-18T00:00:00.000Z,2026-07-06T22:02:39.200Z
161,578,22038764.30,1159132.34,20879631.96,Aura Product 161,Home & Kitchen,Nova,38129.35,2025-11-25T00:00:00.000Z,2026-07-06T22:02:39.200Z
18,778,47121476.10,2658910.77,44462565.33,Aura Product 18,Electronics,Apex,60567.45,2026-04-03T00:00:00.000Z,2026-07-06T22:02:39.200Z
475,596,41854481.44,2429807.18,39424674.26,Aura Product 475,Sports,Apex,70225.64,2025-06-25T00:00:00.000Z,2026-07-06T22:02:39.200Z
12,668,32821044.40,1707382.26,31113662.14,Nova Product 12,Electronics,Nova,49133.30,2026-01-18T00:00:00.000Z,2026-07-06T22:02:39.200Z
434,612,9474616.80,496178.87,8978437.93,Nova Product 434,Sports,Prime,15481.40,2026-04-10T00:00:00.000Z,2026-07-06T22:02:39.200Z
317,595,29710682.40,1403143.15,28307539.25,Orbit Product 317,Beauty,Apex,49933.92,2026-02-18T00:00:00.000Z,2026-07-06T22:02:39.200Z
190,528,17566449.12,888303.55,16678145.57,Aura Product 190,Home & Kitchen,Orbit,33269.79,2025-01-25T00:00:00.000Z,2026-07-06T22:02:39.200Z
225,580,44727204.60,2325043.49,42402161.11,Prime Product 225,Fashion,Prime,77115.87,2026-05-10T00:00:00.000Z,2026-07-06T22:02:39.200Z
70,585,51423155.55,2821680.80,48601474.75,Stride Product 70,Electronics,Nova,87902.83,2025-01-01T00:00:00.000Z,2026-07-06T22:02:39.200Z


####Product Sales Summary to Gold Layer

In [0]:
# Save Product Sales Summary to Gold Layer

(
    product_sales.write
    .mode("overwrite")
    .parquet(f"{gold_base_path}/product_sales/")
)

print("✓ Product Sales Summary written successfully to Gold")

✓ Product Sales Summary written successfully to Gold


####Verify the saved Gold table

In [0]:
# Verify Product Sales Summary from Gold Layer

product_sales_gold = spark.read.parquet(
    f"{gold_base_path}/product_sales/"
)

print("PRODUCT SALES GOLD VERIFICATION")
print("=" * 40)
print(f"Row count: {product_sales_gold.count():,}")

product_sales_gold.printSchema()

PRODUCT SALES GOLD VERIFICATION
Row count: 500
root
 |-- product_id: integer (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_sales: decimal(35,2) (nullable = true)
 |-- total_discount: decimal(23,2) (nullable = true)
 |-- net_sales: decimal(37,2) (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



###Store Sales Summary

Business-ready aggregated table showing sales performance by store.

In [0]:
# Create Store Sales Summary

from pyspark.sql.functions import sum, count, round

store_sales = (
    transactions_silver
    .groupBy("store_id")
    .agg(
        count("transaction_id").alias("total_transactions"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 2).alias("total_sales"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_amount"), 2).alias("net_sales")
    )
)

store_sales = (
    store_sales.join(
        stores_silver,
        on="store_id",
        how="left"
    )
)

store_sales.display()

store_id,total_transactions,total_quantity,total_sales,total_discount,net_sales,store_name,city,state,region,opening_date,last_modified_date,silver_processed_at
18,1936,5713,260479315.10,14760724.74,245718590.36,RetailHub Kolkata 18,Kolkata,West Bengal,East,2020-01-15T00:00:00.000Z,2025-07-22T00:00:00.000Z,2026-07-06T22:08:49.815Z
38,1997,5937,277778954.44,16072306.86,261706647.58,RetailHub Kolkata 38,Kolkata,West Bengal,East,2015-02-01T00:00:00.000Z,2025-06-16T00:00:00.000Z,2026-07-06T22:08:49.815Z
12,1970,5798,270558359.02,14492837.79,256065521.23,RetailHub Mumbai 12,Mumbai,Maharashtra,West,2019-09-21T00:00:00.000Z,2025-03-07T00:00:00.000Z,2026-07-06T22:08:49.815Z
16,2051,6216,278734373.06,15717493.70,263016879.36,RetailHub Mumbai 16,Mumbai,Maharashtra,West,2018-12-06T00:00:00.000Z,2025-12-30T00:00:00.000Z,2026-07-06T22:08:49.815Z
10,1956,5895,266989116.09,15493720.15,251495395.94,RetailHub Hyderabad 10,Hyderabad,Telangana,South,2018-03-19T00:00:00.000Z,2026-02-15T00:00:00.000Z,2026-07-06T22:08:49.815Z
31,2005,6056,285186886.78,16624300.04,268562586.74,RetailHub Chennai 31,Chennai,Tamil Nadu,South,2024-10-22T00:00:00.000Z,2025-10-04T00:00:00.000Z,2026-07-06T22:08:49.815Z
5,1959,5804,275430773.66,15806042.27,259624731.39,RetailHub Jaipur 05,Jaipur,Rajasthan,North,2023-07-18T00:00:00.000Z,2026-05-28T00:00:00.000Z,2026-07-06T22:08:49.815Z
1,1992,5940,268909789.02,14799715.09,254110073.93,RetailHub Chennai 01,Chennai,Tamil Nadu,South,2021-08-11T00:00:00.000Z,2026-06-26T00:00:00.000Z,2026-07-06T22:08:49.815Z
34,2069,6236,288975679.92,17484496.50,271491183.42,RetailHub Kochi 34,Kochi,Kerala,South,2024-06-26T00:00:00.000Z,2026-05-16T00:00:00.000Z,2026-07-06T22:08:49.815Z
27,2037,6194,285175730.89,16529491.77,268646239.12,RetailHub Bengaluru 27,Bengaluru,Karnataka,South,2015-09-05T00:00:00.000Z,2025-04-03T00:00:00.000Z,2026-07-06T22:08:49.815Z


####Save Store Sales Summary

In [0]:
# Save Store Sales Summary to Gold Layer

(
    store_sales.write
    .mode("overwrite")
    .parquet(f"{gold_base_path}/store_sales/")
)

print("✓ Store Sales Summary written successfully to Gold")

✓ Store Sales Summary written successfully to Gold


####Verify the saved Store Sales table

In [0]:
# Verify Store Sales Summary from Gold Layer

store_sales_gold = spark.read.parquet(
    f"{gold_base_path}/store_sales/"
)

print("STORE SALES GOLD VERIFICATION")
print("=" * 40)
print(f"Row count: {store_sales_gold.count():,}")

store_sales_gold.printSchema()

STORE SALES GOLD VERIFICATION
Row count: 50
root
 |-- store_id: integer (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_sales: decimal(35,2) (nullable = true)
 |-- total_discount: decimal(23,2) (nullable = true)
 |-- net_sales: decimal(37,2) (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- opening_date: timestamp (nullable = true)
 |-- last_modified_date: timestamp (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



###Daily Sales Summary

Business-ready aggregated table showing retail sales performance by date.

In [0]:
# Create Daily Sales Summary

from pyspark.sql.functions import (
    to_date,
    sum,
    count,
    countDistinct,
    round
)

daily_sales = (
    transactions_silver
    .withColumn(
        "sales_date",
        to_date("transaction_date")
    )
    .groupBy("sales_date")
    .agg(
        count("transaction_id").alias("total_transactions"),
        countDistinct("customer_id").alias("unique_customers"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 2).alias("total_sales"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_amount"), 2).alias("net_sales")
    )
    .orderBy("sales_date")
)

daily_sales.display()

sales_date,total_transactions,unique_customers,total_quantity,total_sales,total_discount,net_sales
2024-01-01,100,99,297,14249252.72,757703.44,13491549.28
2024-01-02,111,109,323,14933474.66,939745.63,13993729.03
2024-01-03,103,102,310,13312402.68,684240.05,12628162.63
2024-01-04,122,120,348,16493723.97,832882.51,15660841.46
2024-01-05,115,115,360,17174392.16,912971.40,16261420.76
2024-01-06,98,98,288,14068995.89,684880.18,13384115.71
2024-01-07,129,129,395,18367270.42,854895.64,17512374.78
2024-01-08,117,117,354,14073866.41,686152.73,13387713.68
2024-01-09,108,107,301,13701753.06,899467.74,12802285.32
2024-01-10,102,102,305,12638484.22,635600.08,12002884.14


####Save Daily Sales to Gold

In [0]:
# Save Daily Sales Summary to Gold Layer

(
    daily_sales.write
    .mode("overwrite")
    .parquet(f"{gold_base_path}/daily_sales/")
)

print("✓ Daily Sales Summary written successfully to Gold")

✓ Daily Sales Summary written successfully to Gold


###Customer Sales Summary

Business-ready aggregated table showing purchasing behaviour and lifetime value by customer.

In [0]:
# Create Customer Sales Summary

from pyspark.sql.functions import (
    sum,
    count,
    countDistinct,
    min,
    max,
    round
)

customer_sales = (
    transactions_silver
    .groupBy("customer_id")
    .agg(
        count("transaction_id").alias("total_transactions"),
        countDistinct("product_id").alias("unique_products_purchased"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 2).alias("total_sales"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_amount"), 2).alias("net_sales"),
        min("transaction_date").alias("first_purchase_date"),
        max("transaction_date").alias("last_purchase_date")
    )
)

customer_sales = (
    customer_sales.join(
        customers_silver,
        on="customer_id",
        how="left"
    )
)

customer_sales.display()

customer_id,total_transactions,unique_products_purchased,total_quantity,total_sales,total_discount,net_sales,first_purchase_date,last_purchase_date,customer_name,email,city,state,registration_date,last_modified_date,silver_processed_at
782,19,19,55,1808337.51,137047.94,1671289.57,2024-01-15T08:16:03.000Z,2026-06-06T16:46:34.000Z,Riya Chopra,customer782@example.com,Hyderabad,Telangana,2022-07-07T00:00:00.000Z,2022-12-22T00:00:00.000Z,2026-07-06T21:57:22.267Z
2728,20,20,60,3226412.14,192561.97,3033850.17,2024-02-09T21:20:09.000Z,2026-06-13T21:54:55.000Z,Riya Patel,customer2728@example.com,Delhi,Delhi,2023-03-15T00:00:00.000Z,2023-04-22T00:00:00.000Z,2026-07-06T21:57:22.267Z
4376,27,27,76,3569444.73,310533.95,3258910.78,2024-04-16T19:20:11.000Z,2026-06-03T11:29:55.000Z,Arjun Singh,customer4376@example.com,Chennai,Tamil Nadu,2023-08-02T00:00:00.000Z,2023-11-07T00:00:00.000Z,2026-07-06T21:57:22.267Z
3737,21,21,57,2324370.77,126388.77,2197982.00,2024-02-01T02:44:33.000Z,2026-06-15T23:14:05.000Z,Avni Sharma,customer3737@example.com,Pune,Maharashtra,2023-07-15T00:00:00.000Z,2023-12-05T00:00:00.000Z,2026-07-06T21:57:22.267Z
1809,24,24,72,3528042.28,233486.22,3294556.06,2024-02-10T04:47:39.000Z,2026-06-27T06:50:06.000Z,Ishaan Chopra,customer1809@example.com,Delhi,Delhi,2022-03-05T00:00:00.000Z,2022-11-07T00:00:00.000Z,2026-07-06T21:57:22.267Z
1447,25,25,88,3529802.41,125192.51,3404609.90,2024-01-07T06:40:31.000Z,2026-06-22T08:40:22.000Z,Aarav Chopra,customer1447@example.com,Kochi,Kerala,2023-01-04T00:00:00.000Z,2023-07-24T00:00:00.000Z,2026-07-06T21:57:22.267Z
273,27,26,69,3435638.52,231978.93,3203659.59,2024-01-05T09:44:51.000Z,2026-06-21T16:12:29.000Z,Diya Khan,customer273@example.com,Noida,Uttar Pradesh,2022-11-05T00:00:00.000Z,2023-04-11T00:00:00.000Z,2026-07-06T21:57:22.267Z
857,21,21,64,2759058.01,213491.63,2545566.38,2024-02-14T05:29:11.000Z,2026-06-18T09:30:18.000Z,Arjun Reddy,customer857@example.com,Mumbai,Maharashtra,2023-08-30T00:00:00.000Z,2024-03-10T00:00:00.000Z,2026-07-06T21:57:22.267Z
2383,20,19,51,2113291.96,153947.90,1959344.06,2024-01-03T19:00:52.000Z,2026-06-27T08:03:26.000Z,Riya Reddy,customer2383@example.com,Kochi,Kerala,2022-10-31T00:00:00.000Z,2023-01-03T00:00:00.000Z,2026-07-06T21:57:22.267Z
3918,22,22,64,2994378.85,148595.50,2845783.35,2024-01-12T13:56:47.000Z,2026-06-14T07:51:13.000Z,Kiara Reddy,customer3918@example.com,Jaipur,Rajasthan,2022-02-27T00:00:00.000Z,2022-06-16T00:00:00.000Z,2026-07-06T21:57:22.267Z


###Save Customer Sales to Gold

In [0]:
# Save Customer Sales Summary to Gold Layer

(
    customer_sales.write
    .mode("overwrite")
    .parquet(f"{gold_base_path}/customer_sales/")
)

print("✓ Customer Sales Summary written successfully to Gold")

✓ Customer Sales Summary written successfully to Gold


###Final verification of all four Gold tables

In [0]:
# Final Gold Layer Verification

gold_tables = [
    "product_sales",
    "store_sales",
    "daily_sales",
    "customer_sales"
]

print("FINAL GOLD LAYER VERIFICATION")
print("=" * 50)

for table in gold_tables:
    df = spark.read.parquet(f"{gold_base_path}/{table}/")
    print(f"{table:<20} {df.count():>10,} rows")

FINAL GOLD LAYER VERIFICATION
product_sales               500 rows
store_sales                  50 rows
daily_sales                 913 rows
customer_sales            5,000 rows


In [0]:
import requests

token = "<DATABRICKS_TOKEN_REMOVED>"
url = "https://adb-7405619628729867.7.azuredatabricks.net/api/2.1/jobs/run-now"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
body = {"job_id": 659298242613398}

resp = requests.post(url, headers=headers, json=body)
print(resp.status_code)
print(resp.text)

200
{"run_id":232565581942514,"number_in_job":232565581942514}


In [0]:
display(dbutils.fs.ls('abfss://retail@saretailsneh.dfs.core.windows.net/gold'))

path,name,size,modificationTime
abfss://retail@saretailsneh.dfs.core.windows.net/gold/customer_sales/,customer_sales/,0,1783420383000
abfss://retail@saretailsneh.dfs.core.windows.net/gold/daily_sales/,daily_sales/,0,1783419689000
abfss://retail@saretailsneh.dfs.core.windows.net/gold/product_sales/,product_sales/,0,1783419063000
abfss://retail@saretailsneh.dfs.core.windows.net/gold/store_sales/,store_sales/,0,1783419476000


In [0]:
gold_path = "abfss://retail@saretailsneh.dfs.core.windows.net/gold"

for table in ["customer_sales", "daily_sales", "product_sales", "store_sales"]:
    df = spark.read.format("parquet").load(f"{gold_path}/{table}")
    print(f"{table}: {df.count()} rows")

customer_sales: 5000 rows
daily_sales: 913 rows
product_sales: 500 rows
store_sales: 50 rows


In [0]:
gold_daily = spark.read.parquet(
    "abfss://retail@saretailsneh.dfs.core.windows.net/gold/daily_sales/"
)
gold_daily.orderBy(gold_daily.columns[0], ascending=False).show(5)

+----------+------------------+----------------+--------------+-----------+--------------+-----------+
|sales_date|total_transactions|unique_customers|total_quantity|total_sales|total_discount|  net_sales|
+----------+------------------+----------------+--------------+-----------+--------------+-----------+
|2026-07-07|                 1|               1|             2|    3600.00|        100.00|    3500.00|
|2026-07-06|                 5|               3|             9|   13299.97|        375.00|   12924.97|
|2026-06-30|               109|             107|           337|15652134.78|     516218.77|15135916.01|
|2026-06-29|               112|             111|           335|14151869.14|     832202.48|13319666.66|
|2026-06-28|               106|             105|           319|14260104.77|     866752.23|13393352.54|
+----------+------------------+----------------+--------------+-----------+--------------+-----------+
only showing top 5 rows
